### 3. Acervo
La ultima etapa toma los números de registro y los busca en el acervo del IMPI para obtener la información detallada de cada patente. Para esto, se utiliza la función `get_patent_info` que hace una solicitud a la página del acervo con el número de registro y extrae la información relevante utilizando BeautifulSoup.

In [4]:
# import json

# # Cargar los datos desde el archivo JSON (registration_numbers)
# with open("2.MARCIA_result_pages.json", encoding="utf-8") as f:
#     datos = json.load(f)

# registration_numbers = list({
#     item.get("registrationNumber")
#     for item in datos
#     if item.get("registrationNumber") is not None
# })

# # application_numbers = list({
# #     item.get("applicationNumber")
# #     for item in datos
# #     if item.get("applicationNumber") is not None
# # })

# print(len(registration_numbers)) # 153977 registros a scrapear
# #print(len(application_numbers))

In [5]:
# import requests
# from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
# import warnings
# import urllib3

# # ============================== CONFIGURACIÓN ==============================
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)


# BASE_ACERVO = "https://acervomarcas.impi.gob.mx:8181"
# URL_FORMULARIO = f"{BASE_ACERVO}/marcanet/vistas/common/datos/bsqRegistroCompleto.pgi"


# def obtener_detalle_registro(numero_registro: str) -> dict:

#     session = requests.Session() #

#     # 1. GET → ViewState (view_state es necesario para la POST, consiste en un token que mantiene el estado de la sesión en aplicaciones JavaServer Faces)
#     response = session.get(URL_FORMULARIO, verify=False)
#     soup_form = BeautifulSoup(response.text, "html.parser")
#     view_state_tag = soup_form.find("input", {"name": "javax.faces.ViewState"})
#     view_state = view_state_tag["value"] if view_state_tag else ""

#     # 2. POST
#     payload = {
#         "javax.faces.partial.ajax":    "true",
#         "javax.faces.source":          "frmBsqReg:busquedaId",
#         "javax.faces.partial.execute": "@all",
#         "javax.faces.partial.render":  "frmBsqReg:pnlBsqRegistro frmBsqReg:dlgListaRegNac",
#         "frmBsqReg:busquedaId":        "frmBsqReg:busquedaId",
#         "frmBsqReg":                   "frmBsqReg",
#         "frmBsqReg:registroId":        numero_registro,
#         "javax.faces.ViewState":       view_state
#     }

#     headers = {"Content-Type": "application/x-www-form-urlencoded", "Referer": URL_FORMULARIO}
#     respuesta_post = session.post(URL_FORMULARIO, data=payload, headers=headers, verify=False)

#     # 3. Redirect
#     soup_xml = BeautifulSoup(respuesta_post.text, "html.parser")
#     redirect_tag = soup_xml.find("redirect")
#     if not redirect_tag or not redirect_tag.get("url"):
#         return {"error": "No se encontró redirect"}

#     url_detalle = BASE_ACERVO + redirect_tag["url"]
#     respuesta_detalle = session.get(url_detalle, verify=False)
#     soup = BeautifulSoup(respuesta_detalle.text, "html.parser")

#     # ======================== F: EXTRACCIÓN ========================

#     def extraer_seccion_general(div_id: str) -> dict:
#         div = soup.find("div", {"id": div_id})
#         if not div:
#             return {}

#         seccion_map = {
#             "datosGralsSeccion": "datos_generales",
#             "imagenSeccion":     "imagen",
#             "vienaSeccion":      "imagen",
#             "titularSeccion":    "titular",
#             "estabSeccion":      "establecimiento"
#         }

#         resultado = {}
#         seccion_actual = "datos_generales"

#         for element in div.find_all(True):
#             if element.name == "span" and element.get("id") in seccion_map:
#                 seccion_actual = seccion_map[element["id"]]
#                 continue

#             if element.name != "tr":
#                 continue

#             celdas = element.find_all("td")
#             if len(celdas) < 2:
#                 continue

#             nombre = celdas[0].get_text(strip=True)
#             if not nombre:
#                 continue

#             celda_valor = celdas[1]

#             # ==================== EXPEDIENTE ====================
#             enlace = celda_valor.find("a", class_="ui-commandlink")
#             if enlace and "UCMServlet?info=" in enlace.get("onclick", ""):
#                 onclick = enlace["onclick"]
#                 ruta_relativa = onclick.split("window.open('")[1].split("'")[0]
#                 valor = BASE_ACERVO + ruta_relativa          # concatenar con la base para obtener URL completa

#             # ==================== IMAGEN ====================
#             elif celda_valor.find("img"):
#                 valor = celda_valor.find("img").get("src", "")

#             else:
#                 valor = celda_valor.get_text(strip=True)

#             clave = f"{seccion_actual}_{nombre.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}"
#             resultado[clave] = valor

#         return resultado

#     # ==================== PRODUCTOS Y SERVICIOS   ====================

#     def extraer_productos(div_id: str) -> list:
#         div = soup.find("div", {"id": div_id})
#         if not div:
#             return []
#         productos = []
#         for span_clase in div.find_all("span", id=lambda x: x and "claseProdId" in str(x)):
#             clase = span_clase.get_text(strip=True)
#             id_desc = span_clase["id"].replace("claseProdId", "descProId")
#             desc = div.find("span", {"id": id_desc})
#             productos.append({
#                 "clase": clase,
#                 "descripcion": desc.get_text(strip=True) if desc else ""
#             })
#         return productos

#     # ==================== aPODERADO   ====================

#     def extraer_apoderado(div_id: str) -> dict:
#         div = soup.find("div", {"id": div_id})
#         if not div:
#             return {}
#         resultado = {}
#         for fila in div.find_all("tr"):
#             celdas = fila.find_all("td")
#             if len(celdas) < 2:
#                 continue
#             nombre = celdas[0].get_text(strip=True)
#             if not nombre:
#                 continue
#             valor = celdas[1].get_text(strip=True)
#             clave = f"apoderado_{nombre.replace(' ', '_').replace(':', '')}"
#             resultado[clave] = valor
#         return resultado

#     # ======================== Final r ========================
#     return {
#         "datos_generales": extraer_seccion_general("pnlDetalleGral_content"),
#         "productos_servicios": extraer_productos("frmDetalleExp:pnlDetalleGralListas_content"),
#         "apoderado": extraer_apoderado("frmDetalleExp:dtGrdApoderadosId_content")
#     }

## Pruebas para el deployment esacalado a 153,977 registros.


Flujo:
1. Cargar registration_numbers del JSON
2. Cargar processed_set (los que ya se hicieron)
3. pending = [x for x in registration_numbers if x not in processed_set]
4. Procesar en batches de n registros (por ejemplo, 50)
5. Después de cada registro exitoso → append al .jsonl + añadir al processed_set
6. Después de cada batch → guardar processed_set en disco
7. Si el script se rompe (o se interrumpe con Ctrl+C), al volver a correr continúa exactamente donde se quedó



[] Hay que resolver el problema del redirect cuando un registro aparece 2 veces en las busquedas del acervo

[] API ussage.


-------------------------------
Fallidos: al final sera un script que use :1

[] depurar duplicados (no se por que se generaron algunos registros duplicados, aunque no es un gran problema, pero si es algo que se puede evitar)
[] Agregar algo al final que recarge los registros fallidos(que son por bloqueo o por no encontrar el redirect) y los recarge en el pool json inicial para volver a intentar hacerles scraping(con la funcion principal).


In [6]:
# import json
# from pathlib import Path

# # ==================== RUTAS ====================
# json_path = Path("2.MARCIA_result_pages.json")
# ids_txt_path = Path("6.data_integretation_checkpoint_data/confidence_score_Nan_ids.txt")
# output_json_path = Path("2.6.MARCIA_result_pages_nan.json")

# # ==================== CARGAR LISTA DE IDs QUE SÍ QUEREMOS (los NaN) ====================
# if not ids_txt_path.exists():
#     raise FileNotFoundError(f"No se encontró el archivo: {ids_txt_path}")

# with open(ids_txt_path, "r", encoding="utf-8") as f:
#     nan_ids = {line.strip() for line in f if line.strip()}

# print(f"Se cargaron {len(nan_ids):,} IDs del grupo NaN")

# # ==================== CARGAR EL JSON ORIGINAL ====================
# with open(json_path, "r", encoding="utf-8") as f:
#     data = json.load(f)

# print(f"Total de registros en el JSON original: {len(data):,}")

# # ==================== FILTRAR - Mantener SOLO los que SÍ están en nan_ids ====================
# filtered_data = [record for record in data if record.get("id") in nan_ids]

# print(f"Registros encontrados en el grupo NaN: {len(filtered_data):,}")
# print(f"Registros removidos: {len(data) - len(filtered_data):,}")

# # ==================== GUARDAR JSON FILTRADO ====================
# with open(output_json_path, "w", encoding="utf-8") as f:
#     json.dump(filtered_data, f, ensure_ascii=False, indent=2)

# print(f"\n¡Archivo guardado correctamente!")
# print(f"→ {output_json_path}")
# print(f"Tamaño final: {len(filtered_data):,} registros")

In [9]:
import json
from pathlib import Path

# ==================== RUTAS ====================
json_path = Path("2.MARCIA_result_pages.json")
ids_txt_path = Path("2.8_id_clean_sin_municipio_entidad.txt")
output_json_path = Path("2.8.MARCIA_result_pages_sin_municipio_entidad.json")

# ==================== CARGAR LISTA DE IDs QUE QUEREMOS (los sin municipio/entidad) ====================
if not ids_txt_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {ids_txt_path}")

with open(ids_txt_path, "r", encoding="utf-8") as f:
    ids_deseados = {line.strip() for line in f if line.strip()}

print(f"Se cargaron {len(ids_deseados):,} IDs del archivo 2.8_id_clean_sin_municipio_entidad.txt")

# ==================== CARGAR EL JSON ORIGINAL ====================
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total de registros en el JSON original: {len(data):,}")

# ==================== FILTRAR - Mantener SOLO los que están en el txt ====================
filtered_data = [record for record in data if record.get("id") in ids_deseados]

print(f"Registros encontrados y guardados: {len(filtered_data):,}")
print(f"Registros removidos: {len(data) - len(filtered_data):,}")

# ==================== GUARDAR JSON FILTRADO ====================
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print("\n¡Archivo guardado correctamente!")
print(f"→ {output_json_path}")
print(f"Tamaño final: {len(filtered_data):,} registros")

Se cargaron 1,553 IDs del archivo 2.8_id_clean_sin_municipio_entidad.txt
Total de registros en el JSON original: 153,978
Registros encontrados y guardados: 1,553
Registros removidos: 152,425

¡Archivo guardado correctamente!
→ 2.8.MARCIA_result_pages_sin_municipio_entidad.json
Tamaño final: 1,553 registros


In [ ]:
# import json
# import time
# import random
# from pathlib import Path
# from tqdm import tqdm
# import requests
# from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
# import warnings
# import urllib3

# # ============================== CONFIGURACIÓN ==============================
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# BASE_ACERVO = "https://acervomarcas.impi.gob.mx:8181"

# # ==================== PARÁMETROS ====================
# BATCH_SIZE        = 60
# SLEEP_PER_RECORD  = random.uniform(2, 2.5)
# SLEEP_AFTER_BATCH = 1000
# MAX_RETRIES       = 1
# BACKOFF_FACTOR    = 2

# # ==================== ARCHIVOS ====================
# ORIGINAL_JSON    = Path("2.8.MARCIA_result_pages_sin_municipio_entidad.json")
# RESULTADOS_JSONL = Path("3.detalle_resultados.jsonl")
# PROCESSED_FILE   = Path("3.processed_registros.txt")
# FALLIDOS_JSONL   = Path("3.fallidos.jsonl")


# def obtener_detalle_registro(registration_number: str, application_number: str = None) -> dict:
#     # ==================== TU FUNCIÓN ORIGINAL (sin cambios) ====================
#     session = requests.Session()# Guardar en archivo txt (uno por línea)
# # with open('2.8_id_clean_sin_municipio_entidad.txt', 'w', encoding='utf-8') as f:
# #     for id_clean in lista_id_sin_datos:
# #         f.write(id_clean + '\n')

#     def intentar_con_numero(numero: str, es_application: bool = False) -> dict:
#         if es_application:
#             url_form = f"{BASE_ACERVO}/marcanet/vistas/common/datos/bsqExpedienteCompleto.pgi"
#             campo_id = "frmBsqExp:expedienteId"
#             source   = "frmBsqExp:busquedaId2"
#             render   = "frmBsqExp:pnlBsqExp frmBsqExp:dlgListaExpedientes"
#         else:
#             url_form = f"{BASE_ACERVO}/marcanet/vistas/common/datos/bsqRegistroCompleto.pgi"
#             campo_id = "frmBsqReg:registroId"
#             source   = "frmBsqReg:busquedaId"
#             render   = "frmBsqReg:pnlBsqRegistro frmBsqReg:dlgListaRegNac"

#         response       = session.get(url_form, verify=False)
#         soup_form      = BeautifulSoup(response.text, "html.parser")
#         view_state_tag = soup_form.find("input", {"name": "javax.faces.ViewState"})
#         view_state     = view_state_tag["value"] if view_state_tag else ""

#         payload = {
#             "javax.faces.partial.ajax":    "true",
#             "javax.faces.source":          source,
#             "javax.faces.partial.execute": "@all",
#             "javax.faces.partial.render":  render,
#             source:                        source,
#             "frmBsqReg" if not es_application else "frmBsqExp": "frmBsqReg" if not es_application else "frmBsqExp",
#             campo_id:                      numero,
#             "javax.faces.ViewState":       view_state
#         }
#         headers = {
#             "Content-Type": "application/x-www-form-urlencoded",
#             "Referer":      url_form
#         }

#         respuesta_post = session.post(url_form, data=payload, headers=headers, verify=False)
#         soup_xml       = BeautifulSoup(respuesta_post.text, "html.parser")
#         redirect_tag   = soup_xml.find("redirect")

#         if not redirect_tag or not redirect_tag.get("url"):
#             return {"error": "No se encontró redirect"}

#         url_detalle       = BASE_ACERVO + redirect_tag["url"]
#         respuesta_detalle = session.get(url_detalle, verify=False)
#         soup              = BeautifulSoup(respuesta_detalle.text, "html.parser")

#         def extraer_seccion_general(div_id: str) -> dict:
#             div = soup.find("div", {"id": div_id})
#             if not div: return {}
#             seccion_map = {
#                 "datosGralsSeccion": "datos_generales",
#                 "imagenSeccion":     "imagen",
#                 "vienaSeccion":      "imagen",
#                 "titularSeccion":    "titular",
#                 "estabSeccion":      "establecimiento"
#             }
#             resultado = {}
#             seccion_actual = "datos_generales"
#             for element in div.find_all(True):
#                 if element.name == "span" and element.get("id") in seccion_map:
#                     seccion_actual = seccion_map[element["id"]]
#                     continue
#                 if element.name != "tr": continue
#                 celdas = element.find_all("td")
#                 if len(celdas) < 2: continue
#                 nombre = celdas[0].get_text(strip=True)
#                 if not nombre: continue
#                 celda_valor = celdas[1]
#                 enlace = celda_valor.find("a", class_="ui-commandlink")
#                 if enlace and "UCMServlet?info=" in enlace.get("onclick", ""):
#                     ruta_relativa = enlace["onclick"].split("window.open('")[1].split("'")[0]
#                     valor = BASE_ACERVO + ruta_relativa
#                 elif celda_valor.find("img"):
#                     valor = celda_valor.find("img").get("src", "")
#                 else:
#                     valor = celda_valor.get_text(strip=True)
#                 clave = f"{seccion_actual}_{nombre.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}"
#                 resultado[clave] = valor
#             return resultado

#         def extraer_productos(div_id: str) -> list:
#             div = soup.find("div", {"id": div_id})
#             if not div: return []
#             productos = []
#             for span_clase in div.find_all("span", id=lambda x: x and "claseProdId" in str(x)):
#                 clase = span_clase.get_text(strip=True)
#                 id_desc = span_clase["id"].replace("claseProdId", "descProId")
#                 desc = div.find("span", {"id": id_desc})
#                 productos.append({
#                     "clase": clase,
#                     "descripcion": desc.get_text(strip=True) if desc else ""
#                 })
#             return productos

#         def extraer_apoderado(div_id: str) -> dict:
#             div = soup.find("div", {"id": div_id})
#             if not div: return {}
#             resultado = {}
#             for fila in div.find_all("tr"):
#                 celdas = fila.find_all("td")
#                 if len(celdas) < 2: continue
#                 nombre = celdas[0].get_text(strip=True)
#                 if not nombre: continue
#                 valor = celdas[1].get_text(strip=True)
#                 clave = f"apoderado_{nombre.replace(' ', '_').replace(':', '')}"
#                 resultado[clave] = valor
#             return resultado

#         return {
#             "datos_generales":     extraer_seccion_general("pnlDetalleGral_content"),
#             "productos_servicios": extraer_productos("frmDetalleExp:pnlDetalleGralListas_content"),
#             "apoderado":           extraer_apoderado("frmDetalleExp:dtGrdApoderadosId_content")
#         }

#     resultado = intentar_con_numero(registration_number, es_application=False)

#     if "error" in resultado and application_number and application_number != registration_number:
#         print(f"   → Popup detectado. Reintentando con applicationNumber: {application_number}")
#         resultado = intentar_con_numero(application_number, es_application=True)

#     return resultado


# # ======================== SISTEMA DE PROGRESO ========================
# def cargar_processed() -> set:
#     if PROCESSED_FILE.exists():
#         return {line.strip() for line in PROCESSED_FILE.read_text(encoding="utf-8").splitlines() if line.strip()}
#     return set()


# def guardar_processed(processed: set):
#     PROCESSED_FILE.write_text("\n".join(sorted(processed)) + "\n", encoding="utf-8")


# def append_to_jsonl(data: dict, file_path: Path):
#     with file_path.open("a", encoding="utf-8") as f:
#         f.write(json.dumps(data, ensure_ascii=False) + "\n")


# def cargar_fallidos_set() -> set:
#     if not FALLIDOS_JSONL.exists():
#         return set()
#     fallidos = set()
#     with FALLIDOS_JSONL.open(encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line: continue
#             try:
#                 obj = json.loads(line)
#                 reg = obj.get("registrationNumber")
#                 if reg:
#                     fallidos.add(str(reg))
#             except json.JSONDecodeError:
#                 continue
#     return fallidos


# # ======================== FUNCIÓN MEJORADA DE SINCRONIZACIÓN ========================
# def sincronizar_processed_con_jsonl():
#     """PRIMERA ACCIÓN: Actualiza processed_registros.txt con TODOS los registros reales del JSONL"""
#     if not RESULTADOS_JSONL.exists():
#         print("   → No existe aún 3.detalle_resultados.jsonl")
#         return

#     print("   → Sincronizando 3.processed_registros.txt con 3.detalle_resultados.jsonl...")

#     processed_actual = set()
#     total_lineas = 0
#     parseadas_ok = 0
#     saltadas = 0

#     with RESULTADOS_JSONL.open(encoding="utf-8") as f:
#         for line in f:
#             total_lineas += 1
#             line = line.strip()
#             if not line:
#                 continue
#             try:
#                 registro = json.loads(line)
#                 reg_number = registro.get("registrationNumber")
#                 if reg_number:
#                     processed_actual.add(str(reg_number))
#                     parseadas_ok += 1
#                 else:
#                     saltadas += 1
#             except json.JSONDecodeError:
#                 saltadas += 1
#                 continue

#     print(f"   📊 Estadísticas del JSONL:")
#     print(f"      Total líneas leídas     : {total_lineas:,}")
#     print(f"      Parseadas correctamente : {parseadas_ok:,}")
#     print(f"      Saltadas (error)        : {saltadas:,}")

#     if processed_actual:
#         guardar_processed(processed_actual)
#         print(f"     Sincronizado correctamente: {len(processed_actual):,} registros en processed_registros.txt")
#     else:
#         print("   No se encontraron registros válidos en el JSONL")


# # ======================== EJECUCIÓN ========================
# def ejecutar_scraping():
#     print("Iniciando scraper con sistema de resume automático...\n")
    
#     # ←←← PRIMERA ACCIÓN OBLIGATORIA ←←←
#     sincronizar_processed_con_jsonl()

#     with ORIGINAL_JSON.open(encoding="utf-8") as f:
#         datos = json.load(f)

#     processed            = cargar_processed()
#     fallidos             = cargar_fallidos_set()
#     procesados_al_inicio = len(processed)
#     bytes_totales        = 0

#     pending = []
#     for item in datos:
#         reg = item.get("registrationNumber")
#         app = item.get("applicationNumber")
#         if reg and reg not in processed and reg not in fallidos:
#             pending.append({"registrationNumber": reg, "applicationNumber": app})

#     print(f"Total registros originales : {len(datos):,}")
#     print(f"Ya procesados (exitosos)   : {len(processed):,}")
#     print(f"Fallidos definitivos       : {len(fallidos):,}")
#     print(f"Pendientes por scrapear    : {len(pending):,}\n")

#     if not pending:
#         print("Todos los registros ya fueron procesados.")
#         return

#     start_time_total = time.time()

#     for i in range(0, len(pending), BATCH_SIZE):
#         batch_start = time.time()
#         batch       = pending[i:i + BATCH_SIZE]
#         print(f"\nProcesando batch {i//BATCH_SIZE + 1} de {len(pending)//BATCH_SIZE + 1} ({len(batch)} registros)")

#         for item in tqdm(batch, desc="Registros en este batch", leave=False):
#             reg = item["registrationNumber"]
#             app = item.get("applicationNumber")

#             for intento in range(1, MAX_RETRIES + 1):
#                 try:
#                     detalle = obtener_detalle_registro(reg, app)

#                     if (isinstance(detalle, dict) and "error" in detalle) or \
#                        (not detalle.get("datos_generales") and
#                         not detalle.get("productos_servicios") and
#                         not detalle.get("apoderado")):
#                         raise Exception(detalle.get("error", "Respuesta vacía - posible bloqueo del servidor"))

#                     append_to_jsonl({"registrationNumber": reg, "detalle": detalle}, RESULTADOS_JSONL)
#                     processed.add(reg)
#                     bytes_totales += len(str(detalle).encode("utf-8"))

#                     time.sleep(SLEEP_PER_RECORD + random.uniform(0.3, 1.2))
#                     break

#                 except Exception as e:
#                     if intento == MAX_RETRIES:
#                         print(f"  ✗ Falló definitivamente: {reg} → {e}")
#                         append_to_jsonl({"registrationNumber": reg, "error": str(e)}, FALLIDOS_JSONL)
#                     else:
#                         time.sleep(BACKOFF_FACTOR ** (intento - 1))

#         # Estadísticas
#         batch_time        = time.time() - batch_start
#         total_time        = time.time() - start_time_total
#         nuevos_procesados = len(processed) - procesados_al_inicio
#         velocidad         = nuevos_procesados / total_time if total_time > 0 else 0
#         pendientes_reales = len(pending) - nuevos_procesados
#         estimado_min      = (pendientes_reales / velocidad / 60) if velocidad > 0 else 0

#         if nuevos_procesados > 0:
#             bytes_por_reg = bytes_totales / nuevos_procesados
#             mb_restantes  = (pendientes_reales * bytes_por_reg) / (1024 ** 2)
#         else:
#             mb_restantes  = 0

#         print(f"Batch terminado en {batch_time/60:.1f} min | Velocidad: {velocidad:.1f} reg/min")
#         print(f"Nuevos esta sesión: {nuevos_procesados:,} | Estimado restante: {estimado_min:.0f} min")
#         print(f"Tráfico acumulado: {bytes_totales/1024**2:.1f} MB | Estimado MB restante: {mb_restantes:.0f} MB")

#         guardar_processed(processed)

#         print(f"Pausa de {SLEEP_AFTER_BATCH}s...")
#         time.sleep(SLEEP_AFTER_BATCH)

#     print("\n Scraping completado, guardado automáticamente.")


# # ======================== EJECUCIÓN ========================
# if __name__ == "__main__":
#     try:
#         ejecutar_scraping()
#     except KeyboardInterrupt:
#         print("\n\nInterrumpido. Progreso guardado automáticamente.")
#         guardar_processed(cargar_processed())

Iniciando scraper con sistema de resume automático...

   → Sincronizando 3.processed_registros.txt con 3.detalle_resultados.jsonl...
   📊 Estadísticas del JSONL:
      Total líneas leídas     : 1,653
      Parseadas correctamente : 1,653
      Saltadas (error)        : 0
     Sincronizado correctamente: 1,594 registros en processed_registros.txt
Total registros originales : 1,553
Ya procesados (exitosos)   : 1,594
Fallidos definitivos       : 0
Pendientes por scrapear    : 1,349


Procesando batch 1 de 23 (60 registros)


Registros en este batch:   2%|▏         | 1/60 [00:00<00:27,  2.11it/s]

  ✗ Falló definitivamente: 2501688 → Respuesta vacía - posible bloqueo del servidor


Registros en este batch:   3%|▎         | 2/60 [00:00<00:27,  2.10it/s]

  ✗ Falló definitivamente: 2500352 → Respuesta vacía - posible bloqueo del servidor


...

Registros en este batch:  12%|█▏        | 7/60 [00:03<00:24,  2.12it/s]

  ✗ Falló definitivamente: 2507504 → Respuesta vacía - posible bloqueo del servidor


Registros en este batch:  13%|█▎        | 8/60 [00:03<00:24,  2.12it/s]

  ✗ Falló definitivamente: 2501696 → Respuesta vacía - posible bloqueo del servidor




Interrumpido. Progreso guardado automáticamente.


In [22]:
import json
import time
import random
from pathlib import Path
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
import urllib3

# ============================== CONFIGURACIÓN ==============================
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

BASE_ACERVO = "https://acervomarcas.impi.gob.mx:8181"

# ==================== PARÁMETROS ====================
BATCH_SIZE          = 60
SLEEP_PER_RECORD    = random.uniform(2, 2.5)
SLEEP_AFTER_BATCH   = 40
MAX_RETRIES         = 1
BACKOFF_FACTOR      = 2
MAX_BLOQUEOS_CONSEC = 2

# ==================== ARCHIVOS ====================
ORIGINAL_JSON    = Path("2.8.MARCIA_result_pages_sin_municipio_entidad.json")
RESULTADOS_JSONL = Path("3.detalle_resultados.jsonl")
PROCESSED_FILE   = Path("3.processed_registros.txt")
FALLIDOS_JSONL   = Path("3.fallidos.jsonl")


def obtener_detalle_registro(registration_number: str, application_number: str = None) -> dict:
    session = requests.Session()

    def intentar_con_numero(numero: str, es_application: bool = False) -> dict:
        if es_application:
            url_form = f"{BASE_ACERVO}/marcanet/vistas/common/datos/bsqExpedienteCompleto.pgi"
            campo_id = "frmBsqExp:expedienteId"
            source   = "frmBsqExp:busquedaId2"
            render   = "frmBsqExp:pnlBsqExp frmBsqExp:dlgListaExpedientes"
        else:
            url_form = f"{BASE_ACERVO}/marcanet/vistas/common/datos/bsqRegistroCompleto.pgi"
            campo_id = "frmBsqReg:registroId"
            source   = "frmBsqReg:busquedaId"
            render   = "frmBsqReg:pnlBsqRegistro frmBsqReg:dlgListaRegNac"

        response       = session.get(url_form, verify=False)
        soup_form      = BeautifulSoup(response.text, "html.parser")
        view_state_tag = soup_form.find("input", {"name": "javax.faces.ViewState"})
        view_state     = view_state_tag["value"] if view_state_tag else ""

        payload = {
            "javax.faces.partial.ajax":    "true",
            "javax.faces.source":          source,
            "javax.faces.partial.execute": "@all",
            "javax.faces.partial.render":  render,
            source:                        source,
            "frmBsqReg" if not es_application else "frmBsqExp": "frmBsqReg" if not es_application else "frmBsqExp",
            campo_id:                      numero,
            "javax.faces.ViewState":       view_state
        }
        headers = {
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer":      url_form
        }

        respuesta_post = session.post(url_form, data=payload, headers=headers, verify=False)
        soup_xml       = BeautifulSoup(respuesta_post.text, "html.parser")
        redirect_tag   = soup_xml.find("redirect")

        if not redirect_tag or not redirect_tag.get("url"):
            return {"error": "No se encontró redirect"}

        url_detalle       = BASE_ACERVO + redirect_tag["url"]
        respuesta_detalle = session.get(url_detalle, verify=False)
        soup              = BeautifulSoup(respuesta_detalle.text, "html.parser")

        def extraer_seccion_general(div_id: str) -> dict:
            div = soup.find("div", {"id": div_id})
            if not div: return {}
            seccion_map = {
                "datosGralsSeccion": "datos_generales",
                "imagenSeccion":     "imagen",
                "vienaSeccion":      "imagen",
                "titularSeccion":    "titular",
                "estabSeccion":      "establecimiento"
            }
            resultado      = {}
            seccion_actual = "datos_generales"
            for element in div.find_all(True):
                if element.name == "span" and element.get("id") in seccion_map:
                    seccion_actual = seccion_map[element["id"]]
                    continue
                if element.name != "tr": continue
                celdas = element.find_all("td")
                if len(celdas) < 2: continue
                nombre = celdas[0].get_text(strip=True)
                if not nombre: continue
                celda_valor = celdas[1]
                enlace = celda_valor.find("a", class_="ui-commandlink")
                if enlace and "UCMServlet?info=" in enlace.get("onclick", ""):
                    ruta_relativa = enlace["onclick"].split("window.open('")[1].split("'")[0]
                    valor = BASE_ACERVO + ruta_relativa
                elif celda_valor.find("img"):
                    valor = celda_valor.find("img").get("src", "")
                else:
                    valor = celda_valor.get_text(strip=True)
                clave = f"{seccion_actual}_{nombre.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}"
                resultado[clave] = valor
            return resultado

        def extraer_productos(div_id: str) -> list:
            div = soup.find("div", {"id": div_id})
            if not div: return []
            productos = []
            for span_clase in div.find_all("span", id=lambda x: x and "claseProdId" in str(x)):
                clase  = span_clase.get_text(strip=True)
                id_desc = span_clase["id"].replace("claseProdId", "descProId")
                desc   = div.find("span", {"id": id_desc})
                productos.append({
                    "clase":       clase,
                    "descripcion": desc.get_text(strip=True) if desc else ""
                })
            return productos

        def extraer_apoderado(div_id: str) -> dict:
            div = soup.find("div", {"id": div_id})
            if not div: return {}
            resultado = {}
            for fila in div.find_all("tr"):
                celdas = fila.find_all("td")
                if len(celdas) < 2: continue
                nombre = celdas[0].get_text(strip=True)
                if not nombre: continue
                valor  = celdas[1].get_text(strip=True)
                clave  = f"apoderado_{nombre.replace(' ', '_').replace(':', '')}"
                resultado[clave] = valor
            return resultado

        return {
            "datos_generales":     extraer_seccion_general("pnlDetalleGral_content"),
            "productos_servicios": extraer_productos("frmDetalleExp:pnlDetalleGralListas_content"),
            "apoderado":           extraer_apoderado("frmDetalleExp:dtGrdApoderadosId_content")
        }

    resultado = intentar_con_numero(registration_number, es_application=False)

    if "error" in resultado and application_number and application_number != registration_number:
        print(f"   → Popup detectado. Reintentando con applicationNumber: {application_number}")
        resultado = intentar_con_numero(application_number, es_application=True)

    return resultado


# ======================== SISTEMA DE PROGRESO ========================
def cargar_processed() -> set:
    if PROCESSED_FILE.exists():
        return {line.strip() for line in PROCESSED_FILE.read_text(encoding="utf-8").splitlines() if line.strip()}
    return set()


def guardar_processed(processed: set):
    PROCESSED_FILE.write_text("\n".join(sorted(processed)) + "\n", encoding="utf-8")


def append_to_jsonl(data: dict, file_path: Path):
    with file_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")


def cargar_fallidos_set() -> set:
    if not FALLIDOS_JSONL.exists():
        return set()
    fallidos = set()
    with FALLIDOS_JSONL.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                reg = obj.get("registrationNumber")
                if reg:
                    fallidos.add(str(reg))
            except json.JSONDecodeError:
                continue
    return fallidos


def esperar_21_min(reg: str):
    pausa = 21 * 60
    print(f"\n   Bloqueo detectado en {reg}.")
    print(f"   Esperando 21 minutos antes de reintentar...")
    for remaining in range(pausa, 0, -60):
        print(f"    {remaining // 60} min restantes...", flush=True)
        time.sleep(60)
    print(f"   Reintentando {reg}...")


# ======================== SINCRONIZACIÓN ========================
def sincronizar_processed_con_jsonl():
    if not RESULTADOS_JSONL.exists():
        print("   → No existe aún 3.detalle_resultados.jsonl")
        return

    print("   → Sincronizando 3.processed_registros.txt con 3.detalle_resultados.jsonl...")

    processed_actual = set()
    total_lineas = parseadas_ok = saltadas = 0

    with RESULTADOS_JSONL.open(encoding="utf-8") as f:
        for line in f:
            total_lineas += 1
            line = line.strip()
            if not line: continue
            try:
                registro   = json.loads(line)
                reg_number = registro.get("registrationNumber")
                if reg_number:
                    processed_actual.add(str(reg_number))
                    parseadas_ok += 1
                else:
                    saltadas += 1
            except json.JSONDecodeError:
                saltadas += 1
                continue

    print(f"      Estadísticas del JSONL:")
    print(f"      Total líneas leídas     : {total_lineas:,}")
    print(f"      Parseadas correctamente : {parseadas_ok:,}")
    print(f"      Saltadas (error)        : {saltadas:,}")

    if processed_actual:
        guardar_processed(processed_actual)
        print(f"     Sincronizado correctamente: {len(processed_actual):,} registros en processed_registros.txt")
    else:
        print("   No se encontraron registros válidos en el JSONL")


# ======================== EJECUCIÓN ========================
def es_respuesta_vacia(detalle: dict) -> bool:
    return (
        (isinstance(detalle, dict) and "error" in detalle) or
        (not detalle.get("datos_generales") and
         not detalle.get("productos_servicios") and
         not detalle.get("apoderado"))
    )


def ejecutar_scraping():
    print("Iniciando scraper con sistema de resume automático...\n")
    sincronizar_processed_con_jsonl()

    with ORIGINAL_JSON.open(encoding="utf-8") as f:
        datos = json.load(f)

    processed            = cargar_processed()
    fallidos             = cargar_fallidos_set()
    procesados_al_inicio = len(processed)
    bytes_totales        = 0

    pending = []
    for item in datos:
        reg = item.get("registrationNumber")
        app = item.get("applicationNumber")
        if reg and reg not in processed and reg not in fallidos:
            pending.append({"registrationNumber": reg, "applicationNumber": app})

    print(f"Total registros originales : {len(datos):,}")
    print(f"Ya procesados (exitosos)   : {len(processed):,}")
    print(f"Fallidos definitivos       : {len(fallidos):,}")
    print(f"Pendientes por scrapear    : {len(pending):,}\n")

    if not pending:
        print("Todos los registros ya fueron procesados.")
        return

    start_time_total = time.time()
    parar            = False

    for i in range(0, len(pending), BATCH_SIZE):
        if parar:
            break

        batch_start     = time.time()
        batch           = pending[i:i + BATCH_SIZE]
        bloqueos_consec = 0

        print(f"\nProcesando batch {i//BATCH_SIZE + 1} de {len(pending)//BATCH_SIZE + 1} ({len(batch)} registros)")

        for item in tqdm(batch, desc="Registros en este batch", leave=False):
            if parar:
                break

            reg = item["registrationNumber"]
            app = item.get("applicationNumber")

            for intento in range(1, MAX_RETRIES + 1):
                try:
                    detalle = obtener_detalle_registro(reg, app)

                    if es_respuesta_vacia(detalle):
                        raise Exception(detalle.get("error", "Respuesta vacía - posible bloqueo del servidor"))

                    append_to_jsonl({"registrationNumber": reg, "detalle": detalle}, RESULTADOS_JSONL)
                    processed.add(reg)
                    bytes_totales  += len(str(detalle).encode("utf-8"))
                    bloqueos_consec = 0

                    time.sleep(SLEEP_PER_RECORD + random.uniform(0.3, 1.2))
                    break

                except Exception as e:
                    if intento == MAX_RETRIES:
                        bloqueos_consec += 1
                        esperar_21_min(reg)

                        try:
                            detalle = obtener_detalle_registro(reg, app)

                            if es_respuesta_vacia(detalle):
                                raise Exception(detalle.get("error", "Sigue fallando tras espera"))

                            append_to_jsonl({"registrationNumber": reg, "detalle": detalle}, RESULTADOS_JSONL)
                            processed.add(reg)
                            bytes_totales  += len(str(detalle).encode("utf-8"))
                            bloqueos_consec = 0

                            time.sleep(SLEEP_PER_RECORD + random.uniform(0.3, 1.2))

                        except Exception as e2:
                            bloqueos_consec += 1
                            print(f"  ✗ Falló definitivamente tras espera: {reg} → {e2}")
                            append_to_jsonl({"registrationNumber": reg, "error": str(e2)}, FALLIDOS_JSONL)

                            if bloqueos_consec >= MAX_BLOQUEOS_CONSEC * 2:
                                guardar_processed(processed)
                                print(f"\n  {MAX_BLOQUEOS_CONSEC} bloqueos consecutivos sin recuperación.")
                                print(f"  Deteniendo el proceso. Reinicia cuando el servidor esté disponible.")
                                parar = True
                                break
                    else:
                        time.sleep(BACKOFF_FACTOR ** (intento - 1))

        # ── Estadísticas del batch ──────────────────────────────────────────
        batch_time        = time.time() - batch_start
        total_time        = time.time() - start_time_total
        nuevos_procesados = len(processed) - procesados_al_inicio
        velocidad         = nuevos_procesados / total_time if total_time > 0 else 0
        pendientes_reales = len(pending) - nuevos_procesados
        estimado_min      = (pendientes_reales / velocidad / 60) if velocidad > 0 else 0

        bytes_por_reg = bytes_totales / nuevos_procesados if nuevos_procesados > 0 else 0
        mb_restantes  = (pendientes_reales * bytes_por_reg) / (1024 ** 2)

        print(f"Batch terminado en {batch_time/60:.1f} min | Velocidad: {velocidad:.1f} reg/min")
        print(f"Nuevos esta sesión: {nuevos_procesados:,} | Estimado restante: {estimado_min:.0f} min")
        print(f"Tráfico acumulado: {bytes_totales/1024**2:.1f} MB | Estimado MB restante: {mb_restantes:.0f} MB")

        guardar_processed(processed)

        if not parar and i + BATCH_SIZE < len(pending):
            print(f"Pausa de {SLEEP_AFTER_BATCH}s...")
            time.sleep(SLEEP_AFTER_BATCH)

    print("\n Scraping completado, guardado automáticamente.")


# ======================== ENTRY POINT ========================
if __name__ == "__main__":
    try:
        ejecutar_scraping()
    except KeyboardInterrupt:
        print("\n\nInterrumpido. Progreso guardado automáticamente.")
        guardar_processed(cargar_processed())

Iniciando scraper con sistema de resume automático...

   → Sincronizando 3.processed_registros.txt con 3.detalle_resultados.jsonl...
      Estadísticas del JSONL:
      Total líneas leídas     : 2,622
      Parseadas correctamente : 2,622
      Saltadas (error)        : 0
     Sincronizado correctamente: 2,563 registros en processed_registros.txt
Total registros originales : 1,553
Ya procesados (exitosos)   : 2,563
Fallidos definitivos       : 6
Pendientes por scrapear    : 374


Procesando batch 1 de 7 (60 registros)


Registros en este batch:   0%|          | 0/60 [00:00<?, ?it/s]


   Bloqueo detectado en 2600850.
   Esperando 21 minutos antes de reintentar...
    21 min restantes...
    20 min restantes...
    19 min restantes...
    18 min restantes...
    17 min restantes...
    16 min restantes...
    15 min restantes...
    14 min restantes...
    13 min restantes...
    12 min restantes...
    11 min restantes...
    10 min restantes...
    9 min restantes...
    8 min restantes...
    7 min restantes...
    6 min restantes...
    5 min restantes...
    4 min restantes...
    3 min restantes...
    2 min restantes...
    1 min restantes...
   Reintentando 2600850...


Batch terminado en 25.1 min | Velocidad: 0.0 reg/min
Nuevos esta sesión: 60 | Estimado restante: 132 min
Tráfico acumulado: 0.2 MB | Estimado MB restante: 1 MB
Pausa de 40s...

Procesando batch 2 de 7 (60 registros)


Registros en este batch:  67%|██████▋   | 40/60 [02:46<01:19,  3.99s/it]

...

Batch terminado en 4.1 min | Velocidad: 0.1 reg/min
Nuevos esta sesión: 300 | Estimado restante: 21 min
Tráfico acumulado: 0.8 MB | Estimado MB restante: 0 MB
Pausa de 40s...

Procesando batch 6 de 7 (60 registros)


Registros en este batch:   0%|          | 0/60 [00:00<?, ?it/s]


   Bloqueo detectado en 2633433.
   Esperando 21 minutos antes de reintentar...
    21 min restantes...
    20 min restantes...
    19 min restantes...
    18 min restantes...
    17 min restantes...
    16 min restantes...
    15 min restantes...
    14 min restantes...
    13 min restantes...
    12 min restantes...
    11 min restantes...
    10 min restantes...
    9 min restantes...
    8 min restantes...
    7 min restantes...
    6 min restantes...
    5 min restantes...
    4 min restantes...
    3 min restantes...
    2 min restantes...
    1 min restantes...
   Reintentando 2633433...


Batch terminado en 25.2 min | Velocidad: 0.1 reg/min
Nuevos esta sesión: 360 | Estimado restante: 4 min
Tráfico acumulado: 1.0 MB | Estimado MB restante: 0 MB
Pausa de 40s...

Procesando batch 7 de 7 (14 registros)


Batch terminado en 1.0 min | Velocidad: 0.1 reg/min
Nuevos esta sesión: 374 | Estimado restante: 0 min
Tráfico acumulado: 1.0 MB | Estimado MB restante: 0 MB

 Scraping completado, guardado automáticamente.
